In [ ]:
%load_ext autoreload
%autoreload 2

%load_ext tensorboard
%tensorboard --logdir logs/deepdive

print("Autoreload extension loaded. All modules will be reloaded automatically before executing code.")

In [ ]:
import logging
import torch
from torchinfo import summary
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from workspace.config import Config
from workspace.dataset import MNISTDataset
from workspace.model import MNISTClassifier

torch.set_float32_matmul_precision("medium")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = TensorBoardLogger(
    save_dir="data/logs",
    name="deepdive",
)

config = Config()
dataset = MNISTDataset(config)

checkpointer = ModelCheckpoint(
    dirpath=config.checkpoint_rootdir,
    filename="mnist-epoch{epoch:02d}",
    monitor="val_loss",
    mode="min",
    save_last=True,
    save_top_k=1,
)

trainer = pl.Trainer(
    accelerator="cuda",
    devices=[0],
    max_epochs=config.epochs,
    callbacks=[checkpointer],
    deterministic=True,
    enable_progress_bar=True,
    enable_model_summary=False,
    logger=logger,
    precision="16-mixed"
    )


In [ ]:

model = MNISTClassifier(
    layer_depth=config.layer_depth,
    kernel_depth=config.kernel_depth,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
)

print("\nLoaded", config, "\n")
summary(model, input_size=(config.batch_size, 1, 28, 28), verbose=1)

trainer.fit(model, datamodule=dataset)

validation_metrics = trainer.validate(model, datamodule=dataset, ckpt_path="best", verbose=False)
print("Validation metrics:", validation_metrics)
